In [9]:
# from https://techwasti.com/python-tutorial-writing-python-data-class-to-csv
def dataclass_to_csv_row(csv_writer, row):
    csv_writer.writerow(row.__dict__.values())
def dataclass_to_csv_header(csv_writer, header):
    csv_writer.writerow(header.__annotations__.keys())


In [10]:
import xml.etree.ElementTree as ET
from dataclasses import dataclass

@dataclass
class Taxon:
    taxid: int
    name: str
    rank: str

@dataclass
class LineageInfo:
    species: Taxon
    genus: Taxon
    family: Taxon
    order: Taxon
    phylum: Taxon
    kingdom: Taxon
    domain: Taxon

def parse_taxon(taxon: ET.Element):
    """Extract TaxID, rank, scientific name, and human-readable rank from a <Taxon> element."""
    taxid = int(taxon.find("TaxId").text)
    name = str(taxon.find("ScientificName").text)
    rank = str(taxon.find("Rank").text if taxon.find("Rank") is not None else "no rank")
    return Taxon(taxid, name, rank)

def parse_xml(file_path: str, csv_writer):
    """Parse XML file and extract taxonomical information."""
    tree = ET.parse(file_path)
    root = tree.getroot()

    for taxon in root.findall("Taxon"):
        # Extract species-level info
        tax_data = parse_taxon(taxon)
        lineage = taxon.find('Lineage').text
        lin_arr = lineage.split('; ')
        eukayota_idx = lin_arr.index("Eukaryota")
        lin_arr = lin_arr[eukayota_idx:eukayota_idx + 7]
        csv_writer.writerow([tax_data.taxid, tax_data.name, *lin_arr])
""" 
        # Extract lineage info
        lineage = taxon.find("LineageEx")
        if lineage is not None:
            print("\nLineage Information:")
            for ancestor in lineage.findall("Taxon")[-7:]:
                lin_data = parse_taxon(ancestor)
                print(lin_data)
        dataclass_to_csv_row(csv_writer, tax_data) """


' \n        # Extract lineage info\n        lineage = taxon.find("LineageEx")\n        if lineage is not None:\n            print("\nLineage Information:")\n            for ancestor in lineage.findall("Taxon")[-7:]:\n                lin_data = parse_taxon(ancestor)\n                print(lin_data)\n        dataclass_to_csv_row(csv_writer, tax_data) '

In [11]:
import csv

with open('data/tax.csv', 'w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['tax_id', 'species', 'domain', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus'])
    parse_xml('data/taxonomy_result.xml', csv_writer)


In [12]:
import pandas as pd

tax = pd.read_csv('data/tax.csv')
tax

,tax_id,species,domain,kingdom,phylum,class,order,family,genus
0,3387216,Araiospora spinosa,Eukaryota,Sar,Stramenopiles,Oomycota,Rhipidiales,Rhipidiaceae,Araiospora
1,3375946,Phytopythium sp. 'UZ612-like ',Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Phytopythium
2,3375945,Phytopythium sp. IRAN4019C,Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Phytopythium
3,3375944,Phytophthora sp. 'x lacustris-like',Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
4,3375943,Phytophthora sp. 'x Hennops-like1',Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
...,...,...,...,...,...,...,...,...,...
3443,4779,Bremia lactucae,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Bremia
3444,4770,Saprolegnia ferax,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Saprolegnia
3445,4768,Achlya ambisexualis,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Achlya
3446,4767,Achlya klebsiana,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Achlya


In [13]:
tax = tax.dropna(subset='family')
tax = tax[~tax['family'].str.startswith('unclassified')]
tax = tax[~tax['family'].str.startswith('environmental')]
tax = tax.dropna(subset='genus')
tax = tax[~tax['genus'].str.startswith('unclassified')]
tax = tax[~tax['genus'].str.startswith('environmental')]
#tax = tax[~tax['order'].str.contains('incertae sedis')]
#tax = tax[~tax['family'].str.contains('incertae sedis')]
tax

,tax_id,species,domain,kingdom,phylum,class,order,family,genus
0,3387216,Araiospora spinosa,Eukaryota,Sar,Stramenopiles,Oomycota,Rhipidiales,Rhipidiaceae,Araiospora
1,3375946,Phytopythium sp. 'UZ612-like ',Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Phytopythium
2,3375945,Phytopythium sp. IRAN4019C,Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Phytopythium
3,3375944,Phytophthora sp. 'x lacustris-like',Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
4,3375943,Phytophthora sp. 'x Hennops-like1',Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
...,...,...,...,...,...,...,...,...,...
3443,4779,Bremia lactucae,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Bremia
3444,4770,Saprolegnia ferax,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Saprolegnia
3445,4768,Achlya ambisexualis,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Achlya
3446,4767,Achlya klebsiana,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Achlya


In [14]:
import re
def sanitize_org_name(org_name: str) -> str:
    org_name_arr = org_name.split(' ')
    to_remove = ['x', 'var.', 'var', 'aff.', 'aff', 'cf.', 'cf']
    for s in to_remove:
        try:
            org_name_arr.remove(s)
        except ValueError:
            pass
    joined = ' '.join(org_name_arr[:2])
    return re.sub(r"[^a-zA-Z\s\-\.]", "", joined)


In [ ]:
#tax['species'] = tax['species'].apply(sanitize_org_name)
#tax = tax[~tax['species'].str.contains('sp.')]
#tax_unique = tax.copy().drop_duplicates(subset=['species'])
#tax_unique

,tax_id,species,domain,kingdom,phylum,class,order,family,genus
0,3387216,Araiospora spinosa,Eukaryota,Sar,Stramenopiles,Oomycota,Rhipidiales,Rhipidiaceae,Araiospora
1,3375946,Phytopythium sp. 'UZ612-like ',Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Phytopythium
2,3375945,Phytopythium sp. IRAN4019C,Eukaryota,Sar,Stramenopiles,Oomycota,Pythiales,Pythiaceae,Phytopythium
3,3375944,Phytophthora sp. 'x lacustris-like',Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
4,3375943,Phytophthora sp. 'x Hennops-like1',Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Phytophthora
...,...,...,...,...,...,...,...,...,...
3443,4779,Bremia lactucae,Eukaryota,Sar,Stramenopiles,Oomycota,Peronosporales,Peronosporaceae,Bremia
3444,4770,Saprolegnia ferax,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Saprolegnia
3445,4768,Achlya ambisexualis,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Achlya
3446,4767,Achlya klebsiana,Eukaryota,Sar,Stramenopiles,Oomycota,Saprolegniales,Saprolegniaceae,Achlya


In [16]:
tax.to_csv('data/tax-filtered.csv', sep='\t', index=False)

In [3]:
import pandas as pd
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord

In [10]:

records = list(SeqIO.parse('data/sh_general_release_dynamic_all_19.02.2025.fasta', 'fasta'))
def parse_record(record: SeqRecord):
    title = record.id.split('|')
    lineage = title[-1].split(';')
    return lineage
    
records = pd.DataFrame([parse_record(record) for record in records])
records_filtered = records[records[1] == 'p__Oomycota']

In [11]:
records_filtered = records_filtered[~records_filtered[6].str.endswith('_sp')]
records_filtered = records_filtered.drop_duplicates(subset=[6])
records_filtered

,0,1,2,3,4,5,6
28,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Phytophthora,s__Phytophthora_tentaculata
29,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Phytophthora,s__Phytophthora_personensis
520,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Pythiales,f__Pythiaceae,g__Globisporangium,s__Globisporangium_echinulatum
1954,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Pseudoperonospora,s__Pseudoperonospora_cubensis
2108,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Peronospora,s__Peronospora_trifolii-minoris
...,...,...,...,...,...,...,...
155224,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Phytophthora,s__Phytophthora_medicaginis
155225,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Phytophthora,s__Phytophthora_uliginosa
155226,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Phytophthora,s__Phytophthora_gibbosa
155227,k__Stramenopila,p__Oomycota,c__Oomycetes,o__Peronosporales,f__Peronosporaceae,g__Phytophthora,s__Phytophthora_plurivora


In [12]:
records_filtered[3].value_counts()

3
o__Peronosporales    207
o__Pythiales         142
o__Saprolegniales     45
o__Albuginales        13
o__Lagenidiales        4
o__Leptomitales        2
Name: count, dtype: int64